# <a name="seccion5"></a>05. Paso de Modelo E-R a Modelo Relacional

<img src="Modelo Relacional sin Normalizar.png" alt="Diagrama Modelo Relacional sin normalizar|" width="1200px">

Al realizar la transformación del Modelo Entidad-Relación Extendido al Modelo Relacional, se aplicaron mejoras para garantizar la viabilidad del sistema en un entorno real de negocio antes de realizar el proceso de normalización:

1. **Clave Primaria de Vehículos:** Se reemplazó la `Placa` por el `VIN` (Vehicle Identification Number) como Clave Primaria (PK) en la superclase `VEHICULO`. La placa es un atributo que puede cambiar (por pérdida o traslado), mientras que el VIN es inmutable física y legalmente.
2. **Manejo de Vehículos Eléctricos:** Para evitar valores nulos masivos en los vehículos de combustión, atributos específicos como la autonomía y el tiempo de carga se aislaron en la subclase `VEHICULO_ELECTRICO`.
3. **Clave Subrogada en Alquiler:** La agregación central `AlquilerEfectivo` conecta Cliente, Reserva y Vehículo. En lugar de heredar una PK compuesta masiva, se creó el identificador artificial `IdAlquiler` para optimizar la conexión con el `CONTRATO`.
4. **Logística de Talleres y Mantenimiento:** Se independizó el `TALLER` de la `SUCURSAL` para soportar redes de talleres externos. Además, el mantenimiento se dividió en cabecera y detalle para costear piezas exactas sin redundancias.
5. **Control de Reservas:** Se conectó la tabla `RESERVA` directamente con el `CLIENTE` (NIT), un paso lógico omitido en el conceptual, ya que la reserva le pertenece a un usuario antes de volverse un alquiler.

---

### Diccionario de Datos (Modelo Relacional Inicial)

A continuación, se detalla la estructura relacional mapeada. *Nota: Este esquema refleja el estado previo a la normalización, por lo que conserva intencionalmente atributos compuestos y multivaluados del diseño conceptual original.*

#### 1. Módulo de Clientes y Membresías

**Tabla: CLIENTE**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **NIT** | INT | PK | Número de Identificación Tributaria o documento de identidad principal. |
| **CUC** | VARCHAR(20) | - | Código Único de Cliente. Identificador interno del sistema para unificar usuarios. |
| **Nombre** | VARCHAR(100)| - | Nombre completo o razón social del cliente. |
| **Direccion_Compuesta** | VARCHAR(150)| - | Dirección domiciliaria del cliente (Atributo compuesto pendiente de 1FN). |
| **Telefonos** | VARCHAR(100)| - | Números de contacto (Atributo multivaluado pendiente de 1FN). |
| **Recomendador_NIT** | INT | FK | Relación recursiva. Apunta a CLIENTE.NIT indicando quién refirió a este usuario. |

</div>

**Tabla: CLIENTE_OCASIONAL** (Subclase)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **NIT** | INT | PK, FK| Heredado de CLIENTE. |
| **TarjetaCredito** | VARCHAR(20) | - | Tarjeta en garantía para el depósito del alquiler. |
| **LicenciaConduccion**| VARCHAR(20) | - | Número de la licencia de tránsito vigente del conductor. |

</div>

**Tabla: CLIENTE_EMPRESARIAL** (Subclase)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **NIT** | INT | PK, FK| Heredado de CLIENTE. |
| **RazonSocial** | VARCHAR(150)| - | Nombre legal de la empresa para la facturación. |
| **DescuentoCorporativo**| DECIMAL(5,2) | - | Porcentaje de descuento fijo (ej. 15.00) por volumen. |
| **ContactoEjecutivo** | VARCHAR(100)| - | Nombre del encargado de autorizar alquileres en la empresa. |

</div>

**Tabla: MEMBRESIA**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **NombreUnico** | VARCHAR(50) | PK | Nombre identificador del plan (ej. 'Gold', 'Corporate'). |
| **CostoMensual** | DECIMAL(10,2)| - | Tarifa recurrente de la suscripción. |
| **Beneficios** | TEXT | - | Descripción de las ventajas del plan. |

</div>

**Tabla: CONTRATA** (Tabla intermedia M:N)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **NIT** | INT | PK, FK| Relaciona el cliente que adquiere el plan. |
| **NombreUnico** | VARCHAR(50) | PK, FK| Relaciona el plan adquirido. |
| **FechaInicio** | DATE | - | Fecha de inicio de la vigencia de los beneficios. |
| **FechaFin** | DATE | - | Fecha de vencimiento de la membresía. |

</div>

#### 2. Módulo de Flota y Vehículos

**Tabla: VEHICULO** (Superclase)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK | *Vehicle Identification Number*. Identificador físico irrepetible del chasis. |
| **Placa** | VARCHAR(10) | - | Identificador de tránsito (UNIQUE). |
| **Marca** | VARCHAR(50) | - | Fabricante del vehículo. |
| **Modelo** | VARCHAR(50) | - | Línea comercial del vehículo. |
| **Año** | INT | - | Año de fabricación. |
| **KilometrajeActual**| INT | - | Lectura actualizada del odómetro. |
| **TarifaDiariaBase** | DECIMAL(10,2)| - | Costo base del alquiler por día. |
| **NumPasajeros** | INT | - | Capacidad legal de ocupantes. |
| **TipoTransmision** | VARCHAR(20) | - | Dominio: 'Manual', 'Automática'. |
| **TipoTraccion** | VARCHAR(10) | - | Dominio: '4x2', '4x4', 'AWD'. |
| **TipoCombustible** | VARCHAR(20) | - | Dominio: 'Gasolina', 'Diésel', 'Híbrido', 'Eléctrico'. |
| **IdSucursal** | INT | FK | Sucursal base a la que pertenece el vehículo. |

</div>

**Tabla: AUTOMOVIL** (Subclase)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK, FK| Heredado de VEHICULO. |
| **TipoCarroceria** | VARCHAR(30) | - | Dominio: 'Sedán', 'Hatchback', 'SUV'. |

</div>

**Tabla: VAN_CARGA** (Subclase)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK, FK| Heredado de VEHICULO. |
| **CapacidadCargaKg** | DECIMAL(8,2) | - | Peso máximo soportado para logística. |
| **VolumenUtilM3** | DECIMAL(6,2) | - | Espacio en metros cúbicos de la zona de carga. |
| **TipoLicencia** | VARCHAR(5) | - | Restricción legal para conducir (ej. Categoría C1, C2). |

</div>

**Tabla: VEHICULO_ELECTRICO** (Subclase)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK, FK| Heredado de VEHICULO. Aisla atributos eléctricos para evitar NULLs en otros motores. |
| **AutonomiaKm** | INT | - | Kilómetros máximos por carga. |
| **TiempoCargaHoras**| DECIMAL(4,2) | - | Horas necesarias para alcanzar el 100% de batería. |

</div>

#### 3. Módulo de Operaciones (Reservas, Alquileres y Contratos)

**Tabla: RESERVA**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **CodReserva** | INT | PK | Identificador único de la reserva. |
| **NIT_Cliente** | INT | FK | Cliente titular de la reserva. |
| **VIN_Solicitado** | VARCHAR(17) | FK | (Opcional) Vehículo específico bloqueado para esta reserva. |
| **FechaCreacion** | DATE | - | Día en que se procesó la solicitud en sistema. |
| **FechaInicio** | DATE | - | Día proyectado para recoger el vehículo. |
| **FechaFin** | DATE | - | Día proyectado para devolver el vehículo. |
| **Estado** | VARCHAR(20) | - | Dominio: 'Confirmada', 'Cancelada', 'Completada'. |

</div>

**Tabla: ALQUILER_EFECTIVO** (Agregación)
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **IdAlquiler** | INT | PK | Clave subrogada autoincremental para facilitar referencias futuras. |
| **NIT** | INT | FK | Cliente que recibe el vehículo (apunta a CLIENTE). |
| **CodReserva** | INT | FK | Reserva original que originó el alquiler. |
| **VIN** | VARCHAR(17) | FK | Vehículo real entregado (apunta a VEHICULO). |
| **FechaHoraSalida** | DATETIME | - | Marca temporal de la entrega de llaves. |
| **NivelCombSalida** | VARCHAR(20) | - | Nivel de tanque/batería al salir (ej. 'Lleno', '1/2'). |
| **KmSalida** | INT | - | Odómetro de entrega. |
| **FechaHoraLlegada**| DATETIME | - | Marca temporal de la devolución. |
| **NivelCombLlegada**| VARCHAR(20) | - | Nivel de tanque/batería al devolver. |
| **KmLlegada** | INT | - | Odómetro de devolución (para cálculo de excedentes). |
| **NotasEstado** | TEXT | - | Registro de rayones o daños previos/nuevos. |

</div>

**Tabla: CONTRATO**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **NumContrato** | INT | PK | Consecutivo legal y contable del documento. |
| **IdAlquiler** | INT | FK | Apunta a los detalles del evento físico (ALQUILER_EFECTIVO). |
| **FechaFirma** | DATE | - | Fecha de legalización del documento. |
| **Condiciones** | TEXT | - | Cláusulas específicas del servicio. |
| **CostoTotalReal** | DECIMAL(12,2)| - | Liquidación final del servicio (días + extras). |

</div>

**Tabla: POLIZA_SEGURO**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **CodPoliza** | INT | PK | Identificador interno de la póliza contratada. |
| **NumContrato** | INT | FK | Contrato al que ampara el seguro. |
| **Aseguradora** | VARCHAR(100)| - | Compañía externa que emite la cobertura. |
| **TipoCobertura** | VARCHAR(50) | - | Dominio: 'Terceros', 'Todo Riesgo', 'Franquicia'. |
| **CostoDiario** | DECIMAL(10,2)| - | Valor a sumar por cada día de alquiler. |

</div>

#### 4. Módulo de Logística (Sucursales y Mantenimiento)

**Tabla: SUCURSAL**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **IdSucursal** | INT | PK | Identificador de la oficina comercial. |
| **Ciudad** | VARCHAR(50) | - | Municipio de operación. |
| **Direccion** | VARCHAR(150)| - | Ubicación física. |

</div>

**Tabla: TALLER**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **CodTaller** | INT | PK | Identificador del taller (independiente de las sucursales comerciales). |
| **NIT_Taller** | INT | - | Número tributario del taller (si es externo). |
| **NombreTaller** | VARCHAR(100)| - | Razón social o nombre comercial. |
| **TipoTaller** | VARCHAR(20) | - | Dominio: 'Interno', 'Concesionario', 'Aliado Externo'. |
| **Ciudad** | VARCHAR(50) | - | Municipio del taller. |
| **Direccion** | VARCHAR(150)| - | Ubicación física. |

</div>

**Tabla: MANTENIMIENTO**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **IdMantenimiento**| INT | PK | Identificador de la cabecera de entrada a reparación. |
| **VIN** | VARCHAR(17) | FK | Vehículo que ingresa al taller. |
| **CodTaller** | INT | FK | Taller donde se realiza la reparación. |
| **FechaIngreso** | DATE | - | Fecha de entrada. |
| **CostoTotalMant** | DECIMAL(12,2)| - | Valor total facturado de la reparación. |

</div>

**Tabla: DETALLE_MANTENIMIENTO**
<div align="left">

| Atributo | Tipo de Dato | Clave | Descripción|
| :--- | :--- | :--- | :--- |
| **IdDetalle** | INT | PK | Consecutivo de la línea de facturación. |
| **IdMantenimiento**| INT | FK | Apunta a la cabecera del mantenimiento. |
| **RepuestoOServicio**| VARCHAR(100)| - | Pieza instalada o labor cobrada (ej. 'Cambio de Aceite'). |
| **Cantidad** | INT | - | Unidades. |
| **CostoUnitario** | DECIMAL(10,2)| - | Valor por unidad. |
| **Subtotal** | DECIMAL(10,2)| - | Cantidad x CostoUnitario. Permite control exacto de costos de flota. |

</div>

# <a name="seccion6"></a>06. Normalización (1FN, 2FN, 3FN)

<img src="Modelo Relacional Normalizado.png" alt="Diagrama Modelo Relacional Normalizado" width="1200px">

A continuación, se detalla la aplicación de las tres primeras formas normales sobre el diseño relacional inicial.

---

### Paso 1: Primera Forma Normal (1FN)

**Regla Técnica:** Una tabla se encuentra en 1FN si y solo si todos sus dominios simples contienen únicamente valores atómicos; es decir, no existen atributos compuestos ni atributos multivaluados (grupos repetitivos).

**Análisis y Aplicación:**
La entidad `CLIENTE` presentaba dos violaciones directas a esta regla:
1.  **Atributo Compuesto:** El campo `Direccion_Compuesta` almacenaba internamente la ciudad, el barrio y la calle. Si se dejaba así, sería imposible realizar consultas eficientes por ciudad o barrio. Se solucionó dividiendo este campo en tres atributos estrictamente atómicos: `Ciudad`, `Barrio` y `Calle`.
2.  **Atributo Multivaluado:** El campo `Telefonos` permitía almacenar varios números para un mismo cliente, lo que rompe la atomicidad y genera pérdida de integridad de datos. Se solucionó extrayendo este dato a una nueva tabla llamada `CLIENTE TELEFONO`, enlazada mediante la clave foránea `NIT`.

**Evidencia de Transformación:**

*Estructura Original (No normalizada):*
* **CLIENTE** (NIT, CUC, Nombre, **Direccion_Compuesta**, **Telefonos**, Recomendador_NIT)

*Estructura Corregida (1FN):*
* **CLIENTE** (NIT, CUC, Nombre, Ciudad, Barrio, Calle, Recomendador_NIT)
* **CLIENTE TELEFONO** (NIT, Telefono)

---

### Paso 2: Segunda Forma Normal (2FN)

**Regla Técnica:** Una relación está en 2FN si cumple con la 1FN y todos los atributos que no forman parte de la clave primaria dependen *por completo* de ella. Este paso se enfoca en las tablas que tienen **claves primarias compuestas** para descartar dependencias parciales.

**Análisis y Aplicación:**
1.  **Tabla CONTRATA:** Su clave primaria es compuesta `(NIT, NombreUnico)`. Los atributos no clave son `FechaInicio` y `FechaFin`. Estas fechas representan el periodo de vigencia de una membresía específica adquirida por un cliente específico. No dependen solo del `NIT` (el cliente puede tener varias membresías en el tiempo) ni solo del `NombreUnico` (la membresía existe sin clientes). Dependen de la totalidad de la clave.
2.  **Tabla CLIENTE TELEFONO:** Su clave primaria es compuesta `(NIT, Telefono)`. Como no posee atributos adicionales (no clave), cumple automáticamente la regla.

**Conclusión 2FN:** 
El modelo no presenta dependencias parciales. Todas las demás tablas tienen claves primarias simples, por lo que heredan la 2FN automáticamente.

---

### Paso 3: Tercera Forma Normal (3FN)

**Regla Técnica:** Una relación está en 3FN si cumple con la 2FN y cada atributo no clave es mutuamente independiente es decir *no transitivo*. Ningún atributo debe depender funcionalmente de otro atributo que no sea la clave primaria principal.

**Análisis y Aplicación:**
1.  **Independencia del Mantenimiento y los Talleres:**
    Si se incluye la ubicación del taller (`CiudadTaller`, `DireccionTaller`) dentro de la tabla `MANTENIMIENTO`, esos datos dependerían transitivamente de `CodTaller` y no del identificador del mantenimiento. Al crear la tabla independiente **TALLER**, los datos de contacto dependen unicamente de `CodTaller`, cumpliendo la 3FN.
2.  **Jerarquía de Vehículos (Eliminación de Nulos):**
    Si la tabla **VEHICULO** albergara todos los atributos existentes (incluyendo `AutonomiaKm` o `VolumenUtilM3`), generamos dependencias nulas (un automóvil a gasolina no tiene autonomía eléctrica ni volumen de carga). Al modelar mediante herencia y crear las tablas **AUTOMOVIL**, **VAN_CARGA** y **VEHICULO_ELECTRICO**, cada atributo depende de forma exclusiva e intransitiva del `VIN` de su respectiva categoría.
3.  **La Agregación del Alquiler Efectivo:**
    La creación de la tabla central **ALQUILER_EFECTIVO** con la clave `IdAlquiler` absorbió la complejidad relacional de juntar al Cliente, el Vehículo y la Reserva. Gracias a esto, la tabla **CONTRATO** y **POLIZA_SEGURO** mantienen una dependencia directa y limpia de sus propias claves primarias (`NumContrato` y `CodPoliza`), sin arrastrar dependencias transitivas sobre las especificaciones del carro o los datos del usuario.

**Conclusión 3FN:** 
Todos los atributos no clave están completamente relacionados a la clave primaria de su respectiva tabla y son mutuamente independientes. El modelo se encuentra en Tercera Forma Normal (3FN).

### <a name="seccion7"></a>07. Diccionario de Datos Normalizado (Esquema 3FN)

Esquema relacional final y normalizado (3FN)

#### 1. Módulo de Clientes y Membresías

**Tabla: CLIENTE**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **NIT** | INT | PK |
| **CUC** | VARCHAR(20) | - |
| **Nombre** | VARCHAR(100) | - |
| **Ciudad** | VARCHAR(50) | - |
| **Barrio** | VARCHAR(50) | - |
| **Calle** | VARCHAR(100) | - |
| **Recomendador_NIT** | INT | FK |

</div>

**Tabla: CLIENTE_TELEFONO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **NIT** | INT | PK, FK |
| **Telefono** | VARCHAR(20) | PK |

</div>

**Tabla: CLIENTE_OCASIONAL**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **NIT** | INT | PK, FK |
| **TarjetaCredito** | VARCHAR(20) | - |
| **LicenciaConduccion** | VARCHAR(20) | - |

</div>

**Tabla: CLIENTE_EMPRESARIAL**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **NIT** | INT | PK, FK |
| **RazonSocial** | VARCHAR(150) | - |
| **DescuentoCorporativo** | DECIMAL(5,2) | - |
| **ContactoEjecutivo** | VARCHAR(100) | - |

</div>

**Tabla: MEMBRESIA**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **NombreUnico** | VARCHAR(50) | PK |
| **CostoMensual** | DECIMAL(10,2) | - |
| **Beneficios** | TEXT | - |

</div>

**Tabla: CONTRATA**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **NIT** | INT | PK, FK |
| **NombreUnico** | VARCHAR(50) | PK, FK |
| **FechaInicio** | DATE | - |
| **FechaFin** | DATE | - |

</div>

#### 2. Módulo de Flota y Vehículos

**Tabla: VEHICULO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK |
| **Placa** | VARCHAR(10) | - |
| **Marca** | VARCHAR(50) | - |
| **Modelo** | VARCHAR(50) | - |
| **Año** | INT | - |
| **KilometrajeActual** | INT | - |
| **TarifaDiariaBase** | DECIMAL(10,2) | - |
| **NumPasajeros** | INT | - |
| **TipoTransmision** | VARCHAR(20) | - |
| **TipoTraccion** | VARCHAR(10) | - |
| **TipoCombustible** | VARCHAR(20) | - |
| **IdSucursal** | INT | FK |

</div>

**Tabla: AUTOMOVIL**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK, FK |
| **TipoCarroceria** | VARCHAR(30) | - |

</div>

**Tabla: VAN_CARGA**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK, FK |
| **CapacidadCargaKg** | DECIMAL(8,2) | - |
| **VolumenUtilM3** | DECIMAL(6,2) | - |
| **TipoLicencia** | VARCHAR(5) | - |

</div>

**Tabla: VEHICULO_ELECTRICO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK, FK |
| **AutonomiaKm** | INT | - |
| **TiempoCargaHoras** | DECIMAL(4,2) | - |

</div>

**Tabla: VEHICULO_PREMIUM**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **VIN** | VARCHAR(17) | PK, FK |
| **Torque** | VARCHAR(30) | - |
| **Aceleracion** | VARCHAR(30) | - |
| **DetallesLujo** | TEXT | - |

</div>

#### 3. Módulo de Operaciones (Reservas, Alquileres y Contratos)

**Tabla: RESERVA**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **CodReserva** | INT | PK |
| **NIT_Cliente** | INT | FK |
| **VIN_Solicitado** | VARCHAR(17) | FK |
| **FechaCreacion** | DATE | - |
| **FechaInicio** | DATE | - |
| **FechaFin** | DATE | - |
| **Estado** | VARCHAR(20) | - |

</div>

**Tabla: ALQUILER_EFECTIVO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **IdAlquiler** | INT | PK |
| **NIT** | INT | FK |
| **CodReserva** | INT | FK |
| **VIN** | VARCHAR(17) | FK |
| **FechaHoraSalida** | DATETIME | - |
| **NivelCombSalida** | VARCHAR(20) | - |
| **KmSalida** | INT | - |
| **FechaHoraLlegada** | DATETIME | - |
| **NivelCombLlegada** | VARCHAR(20) | - |
| **KmLlegada** | INT | - |
| **NotasEstado** | TEXT | - |

</div>

**Tabla: CONTRATO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **NumContrato** | INT | PK |
| **IdAlquiler** | INT | FK |
| **FechaFirma** | DATE | - |
| **Condiciones** | TEXT | - |
| **CostoTotalReal** | DECIMAL(12,2) | - |

</div>

**Tabla: POLIZA_SEGURO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **CodPoliza** | INT | PK |
| **NumContrato** | INT | FK |
| **Aseguradora** | VARCHAR(100) | - |
| **TipoCobertura** | VARCHAR(50) | - |
| **CostoDiario** | DECIMAL(10,2) | - |

</div>

#### 4. Módulo de Logística y Mantenimiento

**Tabla: SUCURSAL**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **IdSucursal** | INT | PK |
| **Ciudad** | VARCHAR(50) | - |
| **Direccion** | VARCHAR(150) | - |

</div>

**Tabla: TALLER**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **CodTaller** | INT | PK |
| **NIT_Taller** | INT | - |
| **NombreTaller** | VARCHAR(100) | - |
| **TipoTaller** | VARCHAR(20) | - |
| **Ciudad** | VARCHAR(50) | - |
| **Direccion** | VARCHAR(150) | - |

</div>

**Tabla: MANTENIMIENTO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **IdMantenimiento**| INT | PK |
| **VIN** | VARCHAR(17) | FK |
| **CodTaller** | INT | FK |
| **FechaIngreso** | DATE | - |
| **CostoTotalMant** | DECIMAL(12,2) | - |

</div>

**Tabla: DETALLE_MANTENIMIENTO**
<div align="left">

| Atributo | Tipo de Dato | Clave |
| :--- | :--- | :--- |
| **IdDetalle** | INT | PK |
| **IdMantenimiento**| INT | FK |
| **RepuestoOServicio**| VARCHAR(100) | - |
| **Cantidad** | INT | - |
| **CostoUnitario** | DECIMAL(10,2) | - |
| **Subtotal** | DECIMAL(10,2) | - |

</div>